In [13]:
import os
import joblib
import pandas as pd
import numpy as np
import warnings

# بی‌صدا کردن هشدارهای غیرضروری
warnings.filterwarnings('ignore', category=UserWarning)

class ProteinRecommendationPipeline:
    def __init__(self):
        # مسیر پوشه مدل‌ها
        self.models_dir = r"C:\Windows\System32\protein-recommendation-ai\protein-recommendation-ai\models"
        
        self.regressor_path = os.path.join(self.models_dir, 'random_forest_protein_model.pkl')
        self.classifier_path = os.path.join(self.models_dir, 'trained_supplement_classifier.pkl')
        
        if os.path.exists(self.regressor_path) and os.path.exists(self.classifier_path):
            self.regressor = joblib.load(self.regressor_path)
            self.classifier = joblib.load(self.classifier_path)
            print("✨ Both Random Forest Regressor and Classification models loaded successfully!")
        else:
            raise FileNotFoundError(f"Could not find trained models in:\n{self.models_dir}")

    def predict(self, user_data):
        """
        user_data: دیکشنری شامل مشخصات کاربر
        """
        # ۱. ویژگی‌های مدل رگرسیون (۱۰ ویژگی)
        regressor_features = [
            'Age', 'Is_Male', 'Weight_kg', 'Height_cm', 'BMI', 
            'Body_Fat_Percent', 'Lean_Mass_kg', 'Activity_Score', 
            'Daily_Protein_Intake_g', 'Genetic_Score'
        ]
        
        # ۲. ویژگی‌های مدل کلاسیفایر (۹ ویژگی)
        classifier_features = [
            'Age', 'Gender_Encoded', 'Weight_kg', 'Height_cm', 'BMI', 
            'Body_Fat_Percent', 'Lean_Mass_kg', 'Activity_Score', 
            'Genetic_Score'
        ]
        
        # هماهنگ‌سازی نام متغیر جنسیت
        data = user_data.copy()
        if 'Gender_Encoded' not in data and 'Is_Male' in data:
            data['Gender_Encoded'] = data['Is_Male']
            
        # ساخت DataFrame همراه با نام ستون‌ها (برای رفع دقیق هشدار Scikit-Learn)
        df_regressor = pd.DataFrame([data])[regressor_features]
        df_classifier = pd.DataFrame([data])[classifier_features]
        
        # پیش‌بینی میزان پروتئین
        protein_pred_raw = self.regressor.predict(df_regressor)
        protein_pred = protein_pred_raw.flatten()[0] if isinstance(protein_pred_raw, np.ndarray) else protein_pred_raw
            
        # پیش‌بینی نوع مکمل
        supplement_pred = self.classifier.predict(df_classifier)[0]
        
        return {
            "Daily_Protein_Requirement_g": round(float(protein_pred), 2),
            "Recommended_Supplement": supplement_pred
        }

# --- اجرای تست پایپ‌لاین ---
if __name__ == "__main__":
    try:
        pipeline = ProteinRecommendationPipeline()
        
        sample_user = {
            'Age': 28.0,
            'Is_Male': 1,
            'Weight_kg': 82.5,
            'Height_cm': 180.0,
            'BMI': 25.46,
            'Body_Fat_Percent': 14.5,
            'Lean_Mass_kg': 68.2,
            'Activity_Score': 4.0,           
            'Daily_Protein_Intake_g': 95.0,  
            'Genetic_Score': 50.0            
        }
        
        result = pipeline.predict(sample_user)
        
        print("\n" + "="*40)
        print("🎯 AI RECOMMENDATION SYSTEM OUTPUT:")
        print("="*40)
        print(f"🔹 Daily Protein Target : {result['Daily_Protein_Requirement_g']} grams/day")
        print(f"🔹 Suggested Supplement  : {result['Recommended_Supplement']}")
        print("="*40)
        
    except Exception as e:
        print(f"❌ Error running pipeline: {e}")

✨ Both Random Forest Regressor and Classification models loaded successfully!

🎯 AI RECOMMENDATION SYSTEM OUTPUT:
🔹 Daily Protein Target : 124.24 grams/day
🔹 Suggested Supplement  : Whey_Isolate


In [9]:
import ollama

class LocalNutritionalAgent:
    def __init__(self, model_name="qwen3.5"):
        self.model_name = model_name

    def generate_expert_plan(self, user_profile, ml_results):
        system_instruction = "You are a concise nutritionist. Give 3 short bullet points."
        
        user_prompt = f"""
        User Info:
        - Weight: {user_profile.get('Weight_kg', 80)} kg
        - Target Protein: {ml_results['Daily_Protein_Requirement_g']}g/day
        - Supplement: {ml_results['Recommended_Supplement']}
        
        Give 3 short practical diet tips in English.
        """

        try:
            print(f"🤖 Connecting to Ollama ({self.model_name})...\n")
            print("🚀 RESULT:\n")
            
            stream = ollama.chat(
                model=self.model_name,
                messages=[
                    {'role': 'system', 'content': system_instruction},
                    {'role': 'user', 'content': user_prompt},
                ],
                stream=True,
            )

            for chunk in stream:
                print(chunk['message']['content'], end='', flush=True)
                
            print("\n\n✅ Done!")

        except Exception as e:
            print(f"❌ Error running Ollama Agent: {e}")

# --- اجرای تست ---
if __name__ == "__main__":
    agent = LocalNutritionalAgent(model_name="qwen3.5") 
    
    sample_user = {'Weight_kg': 82.5}
    ml_output = {
        "Daily_Protein_Requirement_g": 124.24,
        "Recommended_Supplement": "Whey_Isolate"
    }
    
    agent.generate_expert_plan(sample_user, ml_output)

🤖 Connecting to Ollama (qwen3.5)...

🚀 RESULT:

* Use whey isolate as a fast-digesting boost after workouts to help hit your ~124g target alongside whole foods like chicken or eggs.
* Drink 3+ liters of water daily; adequate hydration is crucial when consuming high doses of protein supplement for digestion and kidney health.
* Pair supplements with fiber-rich carbs, such as oats or rice, at main meals since protein alone may not provide enough satiety energy for an 82kg bodyweight.

✅ Done!
